# 🍎🍌🍊 BÀI TẬP LỚN: PHÂN LOẠI ĐA LỚP & ĐÁNH GIÁ CHẤT LƯỢNG HOA QUẢ (TƯƠI / HỎNG)
**Giai đoạn 3 (v2.0): Huấn luyện trên Google Colab GPU (Có Sao lưu Tự động & Auto-Resume trên Google Drive) và Xuất 12 Biểu đồ Trực quan hóa Trước & Sau khi Train (300 DPI cho Báo cáo)**

---
### Hướng dẫn sử dụng nhanh:
1. Chọn **Runtime -> Change runtime type -> Hardware accelerator: T4 GPU**.
2. Bấm **Runtime -> Run all**.
3. Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần mở lại và bấm **Run all**, Notebook sẽ **tự động khôi phục (Auto-Resume)** từ đúng Epoch vừa dừng (bao gồm cả trạng thái Optimizer, Scheduler, AMP Scaler và cờ Early-Stopping) trên Google Drive!

In [ ]:
# ==============================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE (SAO LƯU TỰ ĐỘNG) & CÀI ĐẶT THƯ VIỆN
# ==============================================================================
!pip install -q kagglehub scikit-learn pandas matplotlib pillow tqdm

import os
import sys
from pathlib import Path
import torch

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP_ROOT = Path('/content/drive/MyDrive/BTL_PLHoa_qua_Backup')
else:
    BACKUP_ROOT = Path('./colab_backup_local')

MODELS_DIR = BACKUP_ROOT / 'models'
FIGURES_DIR = BACKUP_ROOT / 'reports' / 'figures'
MANIFEST_DIR = BACKUP_ROOT / 'manifests'

for d in (MODELS_DIR, FIGURES_DIR, MANIFEST_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[SYSTEM] Thiết bị tính toán: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')
print(f'[BACKUP] Thư mục sao lưu an toàn trên Google Drive: {BACKUP_ROOT.resolve()}')

In [ ]:
# ==============================================================================
# CELL 2: KHỞI TẠO CÁC MODULE CHUẨN CỦA DỰ ÁN (TƯƠNG THÍCH 100% VỚI VS CODE)
# ==============================================================================
import re
import time
import json
import shutil
from collections import Counter
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def set_global_seed(seed: int = 42) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(42)
print('[OK] Đã nạp các thư viện cốt lõi và cố định Random Seed = 42.')

In [ ]:
# ==============================================================================
# CELL 3: PIPELINE TẢI DATASET KAGGLE, LÀM SẠCH, KHỬ TRÙNG LẶP (dHASH) & CHIA TẬP
# ==============================================================================
import kagglehub

# Có thể đổi sang 'swoyam2609/fresh-and-stale-classification' nếu muốn bộ 12 lớp mở rộng
KAGGLE_DATASET_SLUG = 'sriramr/fruits-fresh-and-rotten-for-classification'
IMG_SIZE = 224
BATCH_SIZE = 32

def normalize_class_name(raw_folder_name: str) -> Optional[str]:
    cleaned = raw_folder_name.strip().lower()
    cleaned = re.sub(r'[\s\-]+', '_', cleaned)
    fresh_kw = ('fresh', 'good', 'healthy')
    rotten_kw = ('rotten', 'stale', 'spoiled', 'bad', 'decayed')
    status, fruit_part = None, ''
    for p in fresh_kw:
        if cleaned.startswith(p):
            status, fruit_part = 'fresh', cleaned[len(p):].strip('_')
            break
    if status is None:
        for p in rotten_kw:
            if cleaned.startswith(p):
                status, fruit_part = 'rotten', cleaned[len(p):].strip('_')
                break
    if status is None or not fruit_part:
        for s in fresh_kw:
            if cleaned.endswith(s):
                status, fruit_part = 'fresh', cleaned[:-len(s)].strip('_')
                break
    if status is None or not fruit_part:
        for s in rotten_kw:
            if cleaned.endswith(s):
                status, fruit_part = 'rotten', cleaned[:-len(s)].strip('_')
                break
    if status is None or not fruit_part:
        return None
    plural_map = {
        'apples': 'apple', 'bananas': 'banana', 'oranges': 'orange',
        'mangoes': 'mango', 'mangos': 'mango', 'strawberries': 'strawberry',
        'tomatoes': 'tomato', 'potatoes': 'potato', 'cucumbers': 'cucumber',
        'capsicums': 'capsicum', 'bittergourds': 'bittergourd', 'bitter_gourds': 'bittergourd',
        'grapes': 'grape', 'guavas': 'guava', 'pomegranates': 'pomegranate'
    }
    return f"{status}_{plural_map.get(fruit_part, fruit_part)}"

def load_rgb_image_safe(image_path: str | Path) -> Image.Image:
    with Image.open(image_path) as img:
        img.load()
        if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
            rgba = img.convert('RGBA')
            white_bg = Image.new('RGBA', rgba.size, (255, 255, 255, 255))
            return Image.alpha_composite(white_bg, rgba).convert('RGB')
        return img.convert('RGB')

def compute_dhash(img: Image.Image, hash_size: int = 8) -> str:
    resized = img.convert('L').resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
    pixels = np.asarray(resized, dtype=np.int16)
    diff = pixels[:, 1:] > pixels[:, :-1]
    bit_string = ''.join('1' if b else '0' for b in diff.flatten())
    return f'{int(bit_string, 2):016x}'

def _is_redundant_nested_folder(file_path: Path, root_path: Path) -> bool:
    rel_parts = [p.lower() for p in file_path.relative_to(root_path).parts[:-1]]
    for i in range(len(rel_parts) - 1):
        if rel_parts[i] == rel_parts[i + 1] and rel_parts[i] in {'dataset', 'data', 'fruits', 'images'}:
            return True
    return False

raw_dataset_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET_SLUG))
print(f'[DATASET] Đã tải bộ dữ liệu tại: {raw_dataset_dir}')

manifest_csv = MANIFEST_DIR / 'dataset_split_manifest.csv'

# Khôi phục nhanh từ manifest đã lưu trên Google Drive nếu đường dẫn file còn hợp lệ
loaded_from_cache = False
if manifest_csv.exists():
    cached_df = pd.read_csv(manifest_csv)
    if not cached_df.empty and Path(cached_df.iloc[0]['filepath']).exists():
        df_clean = cached_df
        loaded_from_cache = True
        print(f'[CACHE HIT] Đã nạp nhanh {len(df_clean)} ảnh sạch từ manifest trên Google Drive!')

if not loaded_from_cache:
    records, seen_hashes = [], {}
    stats = {'nested_redundant_skipped': 0, 'total_scanned': 0, 'corrupted_or_small': 0, 'duplicates_removed': 0, 'valid_kept': 0}
    for fp in sorted(raw_dataset_dir.rglob('*')):
        if not fp.is_file() or fp.suffix.lower() not in VALID_EXTENSIONS:
            continue
        if _is_redundant_nested_folder(fp, raw_dataset_dir):
            stats['nested_redundant_skipped'] += 1
            continue
        norm_label = normalize_class_name(fp.parent.name)
        if norm_label is None and fp.parent != raw_dataset_dir:
            norm_label = normalize_class_name(f'{fp.parent.parent.name}_{fp.parent.name}')
        if norm_label is None:
            continue
        stats['total_scanned'] += 1
        try:
            img = load_rgb_image_safe(fp)
            w, h = img.size
            if w < 32 or h < 32:
                stats['corrupted_or_small'] += 1
                continue
            h_val = compute_dhash(img)
            if h_val in seen_hashes:
                stats['duplicates_removed'] += 1
                continue
            seen_hashes[h_val] = str(fp)
            stats['valid_kept'] += 1
            status, fruit_type = norm_label.split('_', 1)
            records.append({'filepath': str(fp.resolve()), 'label': norm_label, 'status': status, 'fruit_type': fruit_type, 'width': w, 'height': h, 'aspect_ratio': round(w / h, 4), 'dhash': h_val})
        except Exception:
            stats['corrupted_or_small'] += 1

    df_clean = pd.DataFrame(records)
    s1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
    train_idx, temp_idx = next(s1.split(df_clean, df_clean['label']))
    temp_df = df_clean.iloc[temp_idx].reset_index(drop=True)
    s2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
    val_sub, test_sub = next(s2.split(temp_df, temp_df['label']))

    df_clean['split'] = 'train'
    df_clean.loc[temp_idx[val_sub], 'split'] = 'val'
    df_clean.loc[temp_idx[test_sub], 'split'] = 'test'
    df_clean.to_csv(manifest_csv, index=False)
    print(f'[CLEANING SUMMARY] {stats}')

print(df_clean.groupby(['label', 'split']).size().unstack(fill_value=0))

In [ ]:
# ==============================================================================
# CELL 4: KHỞI TẠO DATALOADERS & XUẤT 6 BIỂU ĐỒ EDA TRƯỚC KHI TRAIN (300 DPI)
# ==============================================================================
def get_transforms(img_size: int = 224) -> Dict[str, transforms.Compose]:
    resize_dim = int(img_size * 256 / 224)
    train_tfm = transforms.Compose([
        transforms.Resize((resize_dim, resize_dim)),
        transforms.RandomRotation(degrees=20, fill=(255, 255, 255)),
        transforms.RandomResizedCrop(img_size, scale=(0.80, 1.0), ratio=(0.90, 1.10)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.10), value='random'),
    ])
    eval_tfm = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])
    return {'train': train_tfm, 'val': eval_tfm, 'test': eval_tfm}

class FruitFreshnessDataset(Dataset):
    def __init__(self, df: pd.DataFrame, class_to_idx: Dict[str, int], transform=None):
        self.df = df.reset_index(drop=True)
        self.filepaths = self.df['filepath'].tolist()
        self.labels = [class_to_idx[c] for c in self.df['label'].tolist()]
        self.transform = transform
    def __len__(self) -> int:
        return len(self.filepaths)
    def __getitem__(self, idx: int):
        img = load_rgb_image_safe(self.filepaths[idx])
        if self.transform is not None:
            img = self.transform(img)
        return img, self.labels[idx]

class_names = sorted(df_clean['label'].unique().tolist())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for i, c in enumerate(class_names)}
tfms = get_transforms(IMG_SIZE)

dataloaders = {
    split: DataLoader(
        FruitFreshnessDataset(df_clean[df_clean['split'] == split], class_to_idx, tfms[split]),
        batch_size=BATCH_SIZE,
        shuffle=(split == 'train'),
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
    )
    for split in ('train', 'val', 'test')
}

train_sub = df_clean[df_clean['split'] == 'train']
counts = Counter(train_sub['label'].tolist())
class_weights = torch.tensor(
    [len(train_sub) / (len(class_names) * max(counts[c], 1)) for c in class_names],
    dtype=torch.float32, device=DEVICE
)
print('[CLASS WEIGHTS]', {c: round(float(w), 3) for c, w in zip(class_names, class_weights)})

# --- Vẽ và lưu 6 Biểu đồ EDA Trước Khi Train (300 DPI) ---
# Hình 01: Class Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [1.8, 1]})
cls_cnt = df_clean['label'].value_counts().sort_index()
cols = ['#2ecc71' if c.startswith('fresh_') else '#e74c3c' for c in cls_cnt.index]
bars = ax1.barh(cls_cnt.index, cls_cnt.values, color=cols, edgecolor='black', alpha=0.88)
for b in bars:
    ax1.text(b.get_width() + max(cls_cnt.values)*0.01, b.get_y() + b.get_height()/2, f'{int(b.get_width())}', va='center')
ax1.set_xlim(0, max(cls_cnt.values)*1.15)
ax1.set_title('Số lượng ảnh theo từng lớp (Xanh: Tươi | Đỏ: Hỏng)', fontweight='bold')
st_cnt = df_clean['status'].value_counts()
ax2.pie(st_cnt.values, labels=[f'{s.upper()}\n({v})' for s, v in st_cnt.items()], autopct='%1.1f%%', colors=['#2ecc71' if s=='fresh' else '#e74c3c' for s in st_cnt.index], wedgeprops={'edgecolor':'black'})
ax2.set_title(f'Tỷ lệ Fresh vs. Rotten (IR = {cls_cnt.max()/cls_cnt.min():.2f}x)', fontweight='bold')
plt.tight_layout(); fig.savefig(FIGURES_DIR / '01_class_distribution.png', dpi=300, bbox_inches='tight'); plt.show()

# Hình 02: Stratified Split Verification
fig, ax = plt.subplots(figsize=(12, 5.5))
piv = df_clean.pivot_table(index='label', columns='split', values='filepath', aggfunc='count', fill_value=0)[['train', 'val', 'test']]
piv.plot(kind='bar', stacked=True, ax=ax, color=['#3498db', '#f39c12', '#9b59b6'], edgecolor='black')
ax.set_title('Kiểm chứng phân phối Stratified Split (70% Train - 15% Val - 15% Test)', fontweight='bold')
plt.xticks(rotation=30, ha='right'); plt.tight_layout(); fig.savefig(FIGURES_DIR / '02_stratified_split_verification.png', dpi=300, bbox_inches='tight'); plt.show()

# Hình 03: Image Dimensions & Aspect Ratio
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sub_dim = df_clean.sample(n=min(2000, len(df_clean)), random_state=42)
ax1.scatter(sub_dim['width'], sub_dim['height'], c=['#2ecc71' if s=='fresh' else '#e74c3c' for s in sub_dim['status']], alpha=0.5, s=20)
ax1.axvline(IMG_SIZE, color='navy', linestyle='--'); ax1.axhline(IMG_SIZE, color='navy', linestyle='--')
ax1.set_title('Phân tán Kích thước gốc (Width vs. Height)', fontweight='bold')
ax2.hist(df_clean['aspect_ratio'], bins=35, color='#34495e', edgecolor='white')
ax2.axvline(1.0, color='#e74c3c', linestyle='--', linewidth=2)
ax2.set_title('Phân phối Tỷ lệ Khung hình (Aspect Ratio = W / H)', fontweight='bold')
plt.tight_layout(); fig.savefig(FIGURES_DIR / '03_image_dimensions_analysis.png', dpi=300, bbox_inches='tight'); plt.show()

# Hình 04: Raw Samples Comparison
fruits = sorted(df_clean['fruit_type'].unique().tolist())
fig, axes = plt.subplots(2, len(fruits), figsize=(3.2*len(fruits), 6))
if len(fruits) == 1: axes = np.expand_dims(axes, axis=1)
for c_i, fr in enumerate(fruits):
    for r_i, st in enumerate(['fresh', 'rotten']):
        ax = axes[r_i, c_i]
        lbl = f'{st}_{fr}'
        sub = df_clean[df_clean['label'] == lbl]
        if not sub.empty:
            row = sub.iloc[0]
            ax.imshow(load_rgb_image_safe(row['filepath']).resize((224, 224)))
            ax.set_title(f"{lbl}\n({row['width']}x{row['height']}px)", color='#1e8449' if st=='fresh' else '#c0392b', fontsize=10, fontweight='bold')
        ax.axis('off')
plt.tight_layout(); fig.savefig(FIGURES_DIR / '04_raw_samples_comparison.png', dpi=300, bbox_inches='tight'); plt.show()

# Hình 05: Foreground RGB & Brightness Profile
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
f_b, r_b = [], []
f_rgb, r_rgb = {'R':[], 'G':[], 'B':[]}, {'R':[], 'G':[], 'B':[]}
for st, bl, rgbd in [('fresh', f_b, f_rgb), ('rotten', r_b, r_rgb)]:
    for fp in df_clean[df_clean['status']==st].sample(n=min(200, len(df_clean[df_clean['status']==st])), random_state=42)['filepath']:
        arr = np.asarray(load_rgb_image_safe(fp).resize((128, 128)), dtype=np.float32) / 255.0
        mask = ~((arr[:,:,0]>0.94) & (arr[:,:,1]>0.94) & (arr[:,:,2]>0.94))
        px = arr[mask] if np.any(mask) else arr.reshape(-1, 3)
        bl.append(float(px.mean()))
        for idx_c, k in enumerate(('R', 'G', 'B')): rgbd[k].append(float(px[:, idx_c].mean()))
ax1.hist(f_b, bins=25, alpha=0.65, label='Fresh (Quả tươi)', color='#2ecc71', density=True)
ax1.hist(r_b, bins=25, alpha=0.65, label='Rotten (Quả hỏng)', color='#e74c3c', density=True)
ax1.set_title('Độ sáng Vùng Tiền cảnh Quả (Đã tách nền trắng)', fontweight='bold'); ax1.legend()
xp = np.arange(3)
ax2.bar(xp-0.18, [np.mean(f_rgb[k]) for k in ('R','G','B')], 0.35, label='Fresh', color='#2ecc71', edgecolor='black')
ax2.bar(xp+0.18, [np.mean(r_rgb[k]) for k in ('R','G','B')], 0.35, label='Rotten', color='#e74c3c', edgecolor='black')
ax2.set_xticks(xp); ax2.set_xticklabels(['Red', 'Green', 'Blue']); ax2.set_title('Cường độ R-G-B Vùng Tiền cảnh Quả', fontweight='bold'); ax2.legend()
plt.tight_layout(); fig.savefig(FIGURES_DIR / '05_fresh_vs_rotten_color_profile.png', dpi=300, bbox_inches='tight'); plt.show()

# Hình 06: Augmentation Preview
raw_sample = load_rgb_image_safe(df_clean.iloc[0]['filepath'])
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5)); axes = axes.flatten()
axes[0].imshow(raw_sample.resize((IMG_SIZE, IMG_SIZE))); axes[0].set_title(f"Ảnh Gốc [{df_clean.iloc[0]['label']}]", fontweight='bold'); axes[0].axis('off')
mean_np, std_np = np.array(IMAGENET_MEAN).reshape(1,1,3), np.array(IMAGENET_STD).reshape(1,1,3)
for i in range(1, 8):
    aug_t = tfms['train'](raw_sample)
    aug_img = np.clip(aug_t.permute(1,2,0).numpy()*std_np + mean_np, 0, 1)
    axes[i].imshow(aug_img); axes[i].set_title(f'Biến thể Augmentation #{i}'); axes[i].axis('off')
plt.tight_layout(); fig.savefig(FIGURES_DIR / '06_augmentation_pipeline_preview.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ==============================================================================
# CELL 5: ĐỊNH NGHĨA 3 KIẾN TRÚC MÔ HÌNH & GRAD-CAM (TENSOR-HOOK PYTORCH 2.X)
# ==============================================================================
class ConvBlock(nn.Module):
    def __init__(self, in_c: int, out_c: int, pool: bool = True):
        super().__init__()
        layers = [
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=False),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), nn.BatchNorm2d(out_c), nn.ReLU(inplace=False)
        ]
        if pool: layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)

class CustomCNN(nn.Module):
    def __init__(self, num_classes: int, dropout_p: float = 0.4):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32, pool=True), ConvBlock(32, 64, pool=True),
            ConvBlock(64, 128, pool=True), ConvBlock(128, 256, pool=False)
        )
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout_p, inplace=False), nn.Linear(256, 128),
            nn.ReLU(inplace=False), nn.Dropout(dropout_p*0.75, inplace=False), nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.avgpool(self.features(x)))

def create_model(arch_name: str, num_classes: int, pretrained: bool = True) -> nn.Module:
    arch = arch_name.lower()
    if arch == 'custom_cnn': return CustomCNN(num_classes)
    if arch == 'mobilenet_v2':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=False), nn.Linear(m.classifier[1].in_features, num_classes))
        return m
    if arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        m.classifier = nn.Sequential(nn.Dropout(0.3, inplace=False), nn.Linear(m.classifier[1].in_features, num_classes))
        return m
    raise ValueError(arch_name)

def configure_finetuning_stage(model: nn.Module, arch_name: str, stage: int, unfreeze_last_blocks: int = 5):
    if arch_name == 'custom_cnn':
        for p in model.parameters(): p.requires_grad = True
        return
    if stage == 1:
        for p in model.features.parameters(): p.requires_grad = False
        for p in model.classifier.parameters(): p.requires_grad = True
    else:
        cutoff = max(0, len(model.features) - unfreeze_last_blocks)
        for i, blk in enumerate(model.features):
            for p in blk.parameters(): p.requires_grad = (i >= cutoff)
        for p in model.classifier.parameters(): p.requires_grad = True

def freeze_bn_stats_if_needed(model: nn.Module):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and m.weight is not None and not m.weight.requires_grad:
            m.eval()

class GradCAMExplainer:
    """Sử dụng Tensor-level hook (`output.register_hook`) bên trong `forward_hook` an toàn 100% với PyTorch 2.x."""
    def __init__(self, model: nn.Module):
        self.model = model
        self.target_layer = model.features[-1]
    def generate_heatmap(self, input_tensor: torch.Tensor, target_class_idx: Optional[int] = None):
        acts, grads = [], []
        def f_hook(m, inp, out):
            acts.append(out.detach())
            if out.requires_grad:
                out.register_hook(lambda g: grads.append(g.detach()))
        fh = self.target_layer.register_forward_hook(f_hook)
        prev_states = {p: p.requires_grad for p in self.target_layer.parameters()}
        for p in self.target_layer.parameters(): p.requires_grad_(True)
        try:
            self.model.eval(); self.model.zero_grad(set_to_none=True)
            with torch.enable_grad():
                x = input_tensor.clone().detach().requires_grad_(True)
                logits = self.model(x)
                probs = F.softmax(logits, dim=1)[0].detach().cpu().numpy()
                pred_idx = int(np.argmax(probs))
                c_idx = pred_idx if target_class_idx is None else int(target_class_idx)
                logits[0, c_idx].backward()
            w = torch.mean(grads[0][0], dim=(1, 2), keepdim=True)
            cam = F.relu(torch.sum(w * acts[0][0], dim=0))
            if (cam.max() - cam.min()) > 1e-8: cam = (cam - cam.min()) / (cam.max() - cam.min())
            cam_up = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=input_tensor.shape[2:], mode='bilinear', align_corners=False)[0, 0]
            return cam_up.cpu().numpy(), pred_idx, probs
        finally:
            fh.remove()
            for p, st in prev_states.items(): p.requires_grad_(st)
            self.model.zero_grad(set_to_none=True)

def overlay_heatmap(rgb_np: np.ndarray, cam: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    jet = matplotlib.colormaps['jet'](cam)[:, :, :3]
    return np.clip((1.0 - alpha) * rgb_np + alpha * jet, 0.0, 1.0)

print('[OK] Đã khởi tạo kiến trúc 3 mô hình và bộ giải thích Grad-CAM an toàn trên PyTorch 2.x.')

In [ ]:
# ==============================================================================
# CELL 6: HUẤN LUYỆN CẢ 3 MÔ HÌNH (CÓ AUTO-BACKUP & AUTO-RESUME ĐẦY ĐỦ TRẠNG THÁI)
# ==============================================================================
def compute_safety_metrics(cm: np.ndarray, idx_to_class: Dict[int, str]) -> Dict[str, float]:
    f_idx = [i for i, n in idx_to_class.items() if n.startswith('fresh_')]
    r_idx = [i for i, n in idx_to_class.items() if n.startswith('rotten_')]
    tot_r = float(cm[r_idx, :].sum()) if r_idx else 0.0
    r_as_f = float(cm[np.ix_(r_idx, f_idx)].sum()) if (r_idx and f_idx) else 0.0
    tot_f = float(cm[f_idx, :].sum()) if f_idx else 0.0
    f_as_r = float(cm[np.ix_(f_idx, r_idx)].sum()) if (f_idx and r_idx) else 0.0
    return {
        'false_safe_rate_percent': round((r_as_f / tot_r)*100.0, 4) if tot_r > 0 else 0.0,
        'false_rejection_rate_percent': round((f_as_r / tot_f)*100.0, 4) if tot_f > 0 else 0.0,
    }

@torch.no_grad()
def evaluate_loader(model: nn.Module, loader: DataLoader, criterion: nn.Module):
    model.eval()
    loss_sum, total = 0.0, 0
    y_true, y_pred = [], []
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        loss_sum += loss.item() * x.size(0); total += x.size(0)
        y_true.extend(y.cpu().tolist()); y_pred.extend(torch.argmax(logits, dim=1).cpu().tolist())
    acc = accuracy_score(y_true, y_pred) * 100.0
    mp, mr, mf1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    safety = compute_safety_metrics(cm, idx_to_class)
    return {'loss': loss_sum/max(total,1), 'accuracy': acc, 'macro_precision': mp*100, 'macro_recall': mr*100, 'macro_f1': mf1*100, 'confusion_matrix': cm, 'safety_metrics': safety, 'y_true': np.array(y_true), 'y_pred': np.array(y_pred)}

def build_opt_sched(model: nn.Module, arch: str, stage: int, s1_ep: int, tot_ep: int):
    configure_finetuning_stage(model, arch, stage)
    if arch == 'custom_cnn':
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(tot_ep, 1), eta_min=1e-5)
    elif stage == 1:
        opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(s1_ep, 1), eta_min=2e-4)
    else:
        opt = torch.optim.AdamW([
            {'params': [p for p in model.features.parameters() if p.requires_grad], 'lr': 1e-4},
            {'params': [p for p in model.classifier.parameters() if p.requires_grad], 'lr': 5e-4}
        ], weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(tot_ep - s1_ep, 1), eta_min=1e-6)
    return opt, sch

untrained_effnet = create_model('efficientnet_b0', len(class_names), pretrained=True).to(DEVICE)

ARCHITECTURES = ['custom_cnn', 'mobilenet_v2', 'efficientnet_b0']
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 12
PATIENCE = 5

trained_models: Dict[str, nn.Module] = {}
histories: Dict[str, pd.DataFrame] = {}
test_evaluations: Dict[str, Dict[str, Any]] = {}
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
use_amp = (DEVICE.type == 'cuda')

for arch in ARCHITECTURES:
    print('='*85)
    print(f'>>> BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH: {arch.upper()}')
    resume_path = MODELS_DIR / f'{arch}_last_resume.pth'
    best_path = MODELS_DIR / f'{arch}_best_model.pth'
    csv_path = MODELS_DIR / f'{arch}_training_history.csv'

    model = create_model(arch, len(class_names), pretrained=True).to(DEVICE)
    s1_ep = 0 if arch == 'custom_cnn' else STAGE1_EPOCHS
    tot_ep = STAGE1_EPOCHS + STAGE2_EPOCHS
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    start_ep, best_f1, best_acc, best_ep, no_imp = 1, -1.0, -1.0, 0, 0
    hist_list, is_done, saved_st = [], False, None
    if resume_path.exists():
        saved_st = torch.load(resume_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(saved_st['model_state_dict'])
        start_ep = int(saved_st['epoch']) + 1
        best_f1, best_acc, best_ep, no_imp = float(saved_st['best_val_f1']), float(saved_st['best_val_acc']), int(saved_st['best_epoch']), int(saved_st['epochs_no_improve'])
        hist_list = saved_st.get('history_records', [])
        is_done = bool(saved_st.get('training_completed', False))
        if is_done or start_ep > tot_ep:
            print(f'[AUTO-RESUME] Mô hình {arch} đã hoàn tất từ phiên trước (Best Ep {best_ep}, Best F1: {best_f1:.2f}%). Bỏ qua train!')
        else:
            print(f'[AUTO-RESUME] Khôi phục {arch} từ Epoch {start_ep}/{tot_ep} (Best F1: {best_f1:.2f}%, No-Improve: {no_imp})')

    cur_stage, optimizer, scheduler = 0, None, None
    if not is_done and start_ep <= tot_ep:
        for ep in range(start_ep, tot_ep + 1):
            t0 = time.time()
            t_stage = 1 if ep <= s1_ep else 2
            if t_stage != cur_stage or optimizer is None:
                resuming_same_stage = (saved_st is not None and ep == start_ep and int(saved_st.get('stage', 0)) == t_stage)
                cur_stage = t_stage
                optimizer, scheduler = build_opt_sched(model, arch, cur_stage, s1_ep, tot_ep)
                if resuming_same_stage:
                    if 'optimizer_state_dict' in saved_st: optimizer.load_state_dict(saved_st['optimizer_state_dict'])
                    if 'scheduler_state_dict' in saved_st and saved_st['scheduler_state_dict']: scheduler.load_state_dict(saved_st['scheduler_state_dict'])
                    if use_amp and 'scaler_state_dict' in saved_st and saved_st['scaler_state_dict']: scaler.load_state_dict(saved_st['scaler_state_dict'])
                elif cur_stage == 2 and s1_ep > 0:
                    no_imp = 0

            model.train(); freeze_bn_stats_if_needed(model)
            tr_loss_s, tr_corr, tr_tot = 0.0, 0, 0
            for x, y in dataloaders['train']:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', enabled=use_amp):
                    logits = model(x)
                    loss = criterion(logits, y)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                tr_loss_s += loss.item() * x.size(0)
                tr_corr += int((torch.argmax(logits.detach(), dim=1) == y).sum().item())
                tr_tot += x.size(0)
            scheduler.step()

            tr_loss, tr_acc = tr_loss_s / max(tr_tot, 1), (tr_corr / max(tr_tot, 1)) * 100.0
            val_res = evaluate_loader(model, dataloaders['val'], criterion)
            el = time.time() - t0
            rec = {'epoch': ep, 'stage': cur_stage, 'train_loss': round(tr_loss, 5), 'val_loss': round(val_res['loss'], 5), 'train_acc': round(tr_acc, 3), 'val_acc': round(val_res['accuracy'], 3), 'val_macro_f1': round(val_res['macro_f1'], 3), 'val_fsr_percent': val_res['safety_metrics']['false_safe_rate_percent'], 'elapsed_sec': round(el, 2)}
            hist_list.append(rec)
            pd.DataFrame(hist_list).to_csv(csv_path, index=False)

            tag = ''
            if val_res['macro_f1'] > best_f1:
                best_f1, best_acc, best_ep, no_imp = val_res['macro_f1'], val_res['accuracy'], ep, 0
                tag = ' [*BEST SAVED TO DRIVE*]'
                torch.save({'arch_name': arch, 'num_classes': len(class_names), 'class_names': class_names, 'class_to_idx': class_to_idx, 'idx_to_class': idx_to_class, 'img_size': IMG_SIZE, 'norm_mean': IMAGENET_MEAN, 'norm_std': IMAGENET_STD, 'model_state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()}, 'best_epoch': best_ep, 'best_val_acc': best_acc, 'best_val_f1': best_f1}, best_path)
            else:
                no_imp += 1

            stop_early = (cur_stage == 2 and no_imp >= PATIENCE)
            finished_flag = (ep == tot_ep) or stop_early
            torch.save({'arch_name': arch, 'epoch': ep, 'stage': cur_stage, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(), 'scaler_state_dict': scaler.state_dict() if use_amp else {}, 'best_val_f1': best_f1, 'best_val_acc': best_acc, 'best_epoch': best_ep, 'epochs_no_improve': no_imp, 'history_records': hist_list, 'training_completed': finished_flag}, resume_path)
            print(f"Ep [{ep:02d}/{tot_ep:02d}] (S{cur_stage}) | TrLoss: {tr_loss:.4f}, TrAcc: {tr_acc:.2f}% | ValLoss: {val_res['loss']:.4f}, ValAcc: {val_res['accuracy']:.2f}%, ValF1: {val_res['macro_f1']:.2f}%, FSR: {val_res['safety_metrics']['false_safe_rate_percent']:.2f}% | {el:.1f}s{tag}")
            if stop_early:
                print(f'[EARLY STOPPING] Dừng sớm tại Epoch {ep} (Best Epoch = {best_ep}).')
                break

    best_ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_ckpt['model_state_dict'])
    trained_models[arch] = model
    histories[arch] = pd.DataFrame(hist_list)
    test_evaluations[arch] = evaluate_loader(model, dataloaders['test'], criterion)
    best_ckpt['test_metrics'] = {'accuracy': test_evaluations[arch]['accuracy'], 'macro_f1': test_evaluations[arch]['macro_f1'], 'safety_metrics': test_evaluations[arch]['safety_metrics']}
    torch.save(best_ckpt, best_path)

In [ ]:
# ==============================================================================
# CELL 7: XUẤT TRỌN BỘ BIỂU ĐỒ SAU KHI TRAIN (HÌNH 07 -> HÌNH 12, 300 DPI)
# ==============================================================================
def denorm_img(t: torch.Tensor) -> np.ndarray:
    arr = t.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(arr * np.array(IMAGENET_STD).reshape(1,1,3) + np.array(IMAGENET_MEAN).reshape(1,1,3), 0.0, 1.0)

# --- HÌNH 07: ĐỐI CHỨNG TRƯỚC KHI TRAIN VS SAU KHI TRAIN (KÈM GRAD-CAM) ---
sample_x, sample_y = next(iter(dataloaders['test']))
exp_before = GradCAMExplainer(untrained_effnet)
exp_after = GradCAMExplainer(trained_models['efficientnet_b0'])
fig, axes = plt.subplots(4, 3, figsize=(13, 14))
for i in range(4):
    sx = sample_x[i:i+1].to(DEVICE)
    t_idx = int(sample_y[i].item())
    rgb = denorm_img(sample_x[i])
    cb, pb, prb = exp_before.generate_heatmap(sx)
    ca, pa, pra = exp_after.generate_heatmap(sx)
    axes[i,0].imshow(rgb); axes[i,0].set_title(f'Mẫu Test #{i+1}\nThực tế: {idx_to_class[t_idx]}', fontweight='bold'); axes[i,0].axis('off')
    axes[i,1].imshow(overlay_heatmap(rgb, cb)); axes[i,1].set_title(f'[TRƯỚC KHI TRAIN] Dự đoán: {idx_to_class[pb]}\nĐộ tự tin: {prb[pb]*100:.1f}%', color='#27ae60' if pb==t_idx else '#c0392b'); axes[i,1].axis('off')
    axes[i,2].imshow(overlay_heatmap(rgb, ca)); axes[i,2].set_title(f'[SAU KHI TRAIN] Dự đoán: {idx_to_class[pa]}\nĐộ tự tin: {pra[pa]*100:.1f}%', color='#27ae60' if pa==t_idx else '#c0392b', fontweight='bold'); axes[i,2].axis('off')
plt.suptitle('Đối chứng Dự đoán & Bản đồ nhiệt Grad-CAM Trước và Sau khi Huấn luyện (EFFICIENTNET_B0)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); fig.savefig(FIGURES_DIR / '07_before_vs_after_training_predictions.png', dpi=300, bbox_inches='tight'); plt.show()

# --- HÌNH 08: ĐƯỜNG CONG LOSS, ACCURACY & MACRO-F1 CỦA 3 MÔ HÌNH ---
fig, axes = plt.subplots(3, 2, figsize=(14, 12.5))
for r_i, arch in enumerate(ARCHITECTURES):
    hdf = histories[arch]
    ep = hdf['epoch'].values
    axes[r_i,0].plot(ep, hdf['train_loss'], 'o-', label='Train Loss', color='#2980b9')
    axes[r_i,0].plot(ep, hdf['val_loss'], 's--', label='Val Loss', color='#e74c3c')
    if (hdf['stage']==1).any() and (hdf['stage']==2).any():
        axes[r_i,0].axvline(hdf[hdf['stage']==1]['epoch'].max()+0.5, color='gray', linestyle=':', label='Fine-tune Stage 2')
    axes[r_i,0].set_title(f'[{arch.upper()}] Loss Curve', fontweight='bold'); axes[r_i,0].legend(); axes[r_i,0].grid(True, alpha=0.4)
    axes[r_i,1].plot(ep, hdf['train_acc'], 'o-', label='Train Acc (%)', color='#2980b9')
    axes[r_i,1].plot(ep, hdf['val_acc'], 's--', label='Val Acc (%)', color='#27ae60')
    axes[r_i,1].plot(ep, hdf['val_macro_f1'], '^-.', label='Val Macro-F1 (%)', color='#8e44ad')
    if (hdf['stage']==1).any() and (hdf['stage']==2).any():
        axes[r_i,1].axvline(hdf[hdf['stage']==1]['epoch'].max()+0.5, color='gray', linestyle=':')
    b_idx = hdf['val_macro_f1'].idxmax()
    b_ep, b_f1 = hdf.loc[b_idx, 'epoch'], hdf.loc[b_idx, 'val_macro_f1']
    axes[r_i,1].scatter([b_ep], [b_f1], color='gold', edgecolors='black', s=110, zorder=5, label=f'Best F1 ({b_f1:.2f}% @ Ep {b_ep})')
    axes[r_i,1].set_title(f'[{arch.upper()}] Accuracy & Macro-F1 (%)', fontweight='bold'); axes[r_i,1].legend(loc='lower right'); axes[r_i,1].grid(True, alpha=0.4)
plt.tight_layout(); fig.savefig(FIGURES_DIR / '08_training_curves_3models.png', dpi=300, bbox_inches='tight'); plt.show()

best_arch = max(ARCHITECTURES, key=lambda a: test_evaluations[a]['macro_f1'])
best_ev = test_evaluations[best_arch]

# --- HÌNH 09: CONFUSION MATRIX (SỐ LƯỢNG & %) ---
cm_mat = best_ev['confusion_matrix']
cm_pct = cm_mat.astype(np.float32) / np.maximum(cm_mat.sum(axis=1, keepdims=True), 1) * 100.0
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6.8))
for ax, m, ttl, is_p in [(ax1, cm_mat, f'Confusion Matrix (Mẫu) - {best_arch.upper()}', False), (ax2, cm_pct, f'Confusion Matrix (%) - {best_arch.upper()}', True)]:
    im = ax.imshow(m, cmap='Blues'); fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=35, ha='right'); ax.set_yticklabels(class_names)
    ax.set_title(ttl, fontweight='bold')
    for i in range(m.shape[0]):
        for j in range(m.shape[1]):
            ax.text(j, i, f'{m[i,j]:.1f}%' if is_p else f'{int(m[i,j])}', ha='center', va='center', color='white' if m[i,j] > m.max()/2 else 'black', fontsize=8.5)
plt.tight_layout(); fig.savefig(FIGURES_DIR / '09_confusion_matrices_comparison.png', dpi=300, bbox_inches='tight'); plt.show()

# --- HÌNH 10: HEATMAP CLASSIFICATION REPORT TỪNG LỚP ---
rep_d = classification_report(best_ev['y_true'], best_ev['y_pred'], target_names=class_names, output_dict=True, zero_division=0)
rep_df = pd.DataFrame([{'Class': c, 'Precision (%)': rep_d[c]['precision']*100, 'Recall (%)': rep_d[c]['recall']*100, 'F1-Score (%)': rep_d[c]['f1-score']*100, 'Support': int(rep_d[c]['support'])} for c in class_names]).set_index('Class')
rep_df.to_csv(FIGURES_DIR / 'per_class_classification_report.csv')
fig, ax = plt.subplots(figsize=(9, max(5.5, 0.55*len(class_names))))
mv = rep_df[['Precision (%)', 'Recall (%)', 'F1-Score (%)']].values
im = ax.imshow(mv, cmap='YlGn', vmin=60, vmax=100, aspect='auto'); fig.colorbar(im, ax=ax)
ax.set_xticks([0,1,2]); ax.set_xticklabels(['Precision (%)', 'Recall (%)', 'F1-Score (%)'], fontweight='bold')
ax.set_yticks(range(len(class_names))); ax.set_yticklabels([f"{c} (n={rep_df.loc[c,'Support']})" for c in class_names])
for i in range(mv.shape[0]):
    for j in range(mv.shape[1]): ax.text(j, i, f'{mv[i,j]:.2f}%', ha='center', va='center', fontweight='bold')
ax.set_title(f'Classification Report Chi tiết - {best_arch.upper()}', fontweight='bold')
plt.tight_layout(); fig.savefig(FIGURES_DIR / '10_per_class_classification_report.png', dpi=300, bbox_inches='tight'); plt.show()

# --- HÌNH 11: BẢNG & BIỂU ĐỒ SO SÁNH THỰC NGHIỆM 3 MÔ HÌNH (ABLATION STUDY) ---
summary_rows = []
for arch in ARCHITECTURES:
    m = trained_models[arch]
    ev = test_evaluations[arch]
    ckpt_sz = (MODELS_DIR / f'{arch}_best_model.pth').stat().st_size / (1024*1024)
    m_cpu = m.to('cpu').eval()
    dx = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    with torch.no_grad():
        for _ in range(3): _ = m_cpu(dx)
        t0 = time.perf_counter()
        for _ in range(10): _ = m_cpu(dx)
        cpu_ms = ((time.perf_counter() - t0) / 10) * 1000.0
    m.to(DEVICE)
    summary_rows.append({
        'Model': arch.upper(),
        'Test Accuracy (%)': round(ev['accuracy'], 2),
        'Macro Precision (%)': round(ev['macro_precision'], 2),
        'Macro Recall (%)': round(ev['macro_recall'], 2),
        'Macro F1-Score (%)': round(ev['macro_f1'], 2),
        'False Safe Rate - FSR (%)': round(ev['safety_metrics']['false_safe_rate_percent'], 2),
        'False Rejection Rate - FRR (%)': round(ev['safety_metrics']['false_rejection_rate_percent'], 2),
        'CPU Latency (ms/img)': round(cpu_ms, 2),
        'Model Size (MB)': round(ckpt_sz, 2),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(FIGURES_DIR / 'model_comparison_summary.csv', index=False)
display(summary_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.4, 1]})
x = np.arange(len(summary_df)); w = 0.25
b1 = ax1.bar(x-w, summary_df['Test Accuracy (%)'], w, label='Test Accuracy (%)', color='#2980b9', edgecolor='black')
b2 = ax1.bar(x, summary_df['Macro F1-Score (%)'], w, label='Macro F1-Score (%)', color='#27ae60', edgecolor='black')
b3 = ax1.bar(x+w, summary_df['False Safe Rate - FSR (%)'], w, label='False Safe Rate - FSR (%)', color='#e74c3c', edgecolor='black')
for bars in (b1, b2, b3):
    for b in bars: ax1.text(b.get_x()+b.get_width()/2, b.get_height()+1.2, f'{b.get_height():.2f}%', ha='center', fontsize=9, fontweight='bold')
ax1.set_xticks(x); ax1.set_xticklabels(summary_df['Model'], fontweight='bold'); ax1.set_ylim(0, 112); ax1.set_title('So sánh Accuracy, Macro-F1 & False Safe Rate (FSR)', fontweight='bold'); ax1.legend()
bl = ax2.bar(x-0.18, summary_df['CPU Latency (ms/img)'], 0.35, label='CPU Latency (ms)', color='#f39c12', edgecolor='black')
bs = ax2.bar(x+0.18, summary_df['Model Size (MB)'], 0.35, label='Model Size (MB)', color='#8e44ad', edgecolor='black')
for bars in (bl, bs):
    for b in bars: ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.4, f'{b.get_height():.1f}', ha='center', fontsize=9)
ax2.set_xticks(x); ax2.set_xticklabels(summary_df['Model'], fontweight='bold'); ax2.set_title('So sánh Tốc độ Suy luận CPU & Dung lượng Trọng số', fontweight='bold'); ax2.legend()
plt.tight_layout(); fig.savefig(FIGURES_DIR / '11_ablation_model_comparison.png', dpi=300, bbox_inches='tight'); plt.show()

# --- HÌNH 12: GRAD-CAM CA DỰ ĐOÁN ĐÚNG & PHÂN TÍCH CA NHẦM LẪN (ERROR ANALYSIS) ---
best_m = trained_models[best_arch]
expl = GradCAMExplainer(best_m)
corr_list, err_list = [], []
for bx, by in dataloaders['test']:
    for i in range(bx.size(0)):
        sx = bx[i:i+1].to(DEVICE); ti = int(by[i].item())
        with torch.no_grad():
            pr = F.softmax(best_m(sx), dim=1)[0].cpu().numpy()
            pi = int(np.argmax(pr))
        if pi == ti and len(corr_list) < 4: corr_list.append((bx[i], ti))
        elif pi != ti and len(err_list) < 4: err_list.append((bx[i], ti))
    if len(corr_list) >= 4 and len(err_list) >= 4: break
while len(err_list) < 4 and corr_list: err_list.append(corr_list[-1])

fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for r_i, items in enumerate([corr_list[:4], err_list[:4]]):
    for c_i in range(4):
        ax = axes[r_i, c_i]
        if c_i < len(items):
            it, ti = items[c_i]
            cam, pi, pr = expl.generate_heatmap(it.unsqueeze(0).to(DEVICE))
            rgb = denorm_img(it)
            ax.imshow(overlay_heatmap(rgb, cam))
            ok = (ti == pi)
            ax.set_title(f"[{'ĐÚNG' if ok else 'NHẦM LẪN'}] Thực: {idx_to_class[ti]}\nĐoán: {idx_to_class[pi]} ({pr[pi]*100:.1f}%)", color='#1e8449' if ok else '#c0392b', fontsize=9.5, fontweight='bold')
        ax.axis('off')
plt.suptitle(f'Giải thích Mô hình bằng Grad-CAM & Phân tích Lỗi ({best_arch.upper()})', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); fig.savefig(FIGURES_DIR / '12_gradcam_correct_and_errors.png', dpi=300, bbox_inches='tight'); plt.show()

In [ ]:
# ==============================================================================
# CELL 8: ĐÓNG GÓI TOÀN BỘ TRỌNG SỐ (.PTH) & 12 BIỂU ĐỒ 300 DPI THÀNH FILE .ZIP
# ==============================================================================
zip_base = Path('/content/BTL_Colab_Results_Bundle') if IN_COLAB else Path('./BTL_Colab_Results_Bundle')
archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=BACKUP_ROOT)
print(f'[COMPLETED] Đã đóng gói toàn bộ Trọng số & 12 Biểu đồ Báo cáo tại: {archive_path}')
if IN_COLAB:
    from google.colab import files
    files.download(archive_path)